# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedKroush/Flyrank-ML1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb
import pandas as pd
import os
import getpass

# Get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

# Connect to DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Source table
fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

# Build the March 2026 page-level dataset
labeled = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {fact_daily}
        WHERE report_date >= '2026-03-01'
          AND report_date < '2026-04-01'
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last15,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev15,

            AVG(f.gsc_avg_position) AS avg_position_month,

            SUM(f.gsc_impressions) AS impressions_month,
            SUM(f.gsc_clicks) AS clicks_month

        FROM {fact_daily} f, bounds b

        WHERE f.report_date >= '2026-03-01'
          AND f.report_date < '2026-04-01'

        GROUP BY 1, 2

        HAVING imp_prev15 >= 50
    )

    SELECT *,
           (imp_last15 < 0.8 * imp_prev15)::INT AS is_declining
    FROM windowed
""").df()

print(f"Rows loaded: {len(labeled):,}")
print(f"Decline base rate: {labeled['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 94,559
Decline base rate: 0.373


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

The prediction unit is one content page for one client. I use three numeric search-performance features that are available before the prediction point:

* `impressions_month`: total Google Search impressions in the feature window.
* `clicks_month`: total Google Search clicks in the feature window.
* `avg_position_month`: average Google Search position in the feature window.

These features are kept numeric and do not require categorical encoding. Missing numeric values are filled with zero only where the absence represents no recorded activity; otherwise missingness is checked explicitly. Client and content hash IDs are retained only for grouping and joining, never as model features.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the feature vector used by the capstone

feature_cols = [
    "impressions_month",
    "clicks_month",
    "avg_position_month",
]

features = labeled[feature_cols].copy()

# Check the feature columns exist
assert set(feature_cols).issubset(features.columns)

# Numeric conversion
for col in feature_cols:
    features[col] = pd.to_numeric(features[col], errors="coerce")

print("Feature columns:")
print(features.columns.tolist())

print("\nShape:")
print(features.shape)

print("\nMissing values:")
print(features.isna().sum())

print("\nFeature dtypes:")
print(features.dtypes)

Feature columns:
['impressions_month', 'clicks_month', 'avg_position_month']

Shape:
(94559, 3)

Missing values:
impressions_month     0
clicks_month          0
avg_position_month    0
dtype: int64

Feature dtypes:
impressions_month     float64
clicks_month          float64
avg_position_month    float64
dtype: object


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

`impressions_month` measures the number of Google Search impressions observed during the feature window. `clicks_month` measures Google Search clicks during the same available feature window. `avg_position_month` measures average search position.

All three are numeric, so no categorical encoding is required.

The important availability condition is that these features must be calculated from data available **before the outcome is measured**. The later decline outcome must not be used to construct any of these features.

Missing values are inspected rather than silently treated as evidence of decline. Where zero genuinely means that no recorded activity occurred, zero is an appropriate value; otherwise the row should be investigated rather than automatically imputed.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature availability and missingness checks

print("=== FEATURE NOTES / CHECKS ===")

for col in feature_cols:
    print(f"\n{col}")
    print(f"  dtype: {features[col].dtype}")
    print(f"  missing: {features[col].isna().sum():,}")
    print(f"  non-missing: {features[col].notna().sum():,}")

print("\nAll model features are numeric:")
print(features.dtypes.apply(lambda x: pd.api.types.is_numeric_dtype(x)).to_dict())

=== FEATURE NOTES / CHECKS ===

impressions_month
  dtype: float64
  missing: 0
  non-missing: 94,559

clicks_month
  dtype: float64
  missing: 0
  non-missing: 94,559

avg_position_month
  dtype: float64
  missing: 0
  non-missing: 94,559

All model features are numeric:
{'impressions_month': True, 'clicks_month': True, 'avg_position_month': True}


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I specifically checked for three leakage risks.

First, label-derived fields such as `trend_direction`, `trend_pct`, and `is_declining` must not enter the feature vector because they are constructed from the outcome or from information that would only be known after the prediction point.

Second, future-window measurements must not be used as model features. The decline label is based on the later observation period, while the features represent the information available before that outcome.

Third, pseudonymous identifiers such as `client_hash_id` and `content_hash_id` are not predictive features. They are retained only to identify and group observations.

The code below checks the feature list against known label-derived, identifier, and future/outcome-related fields.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage checks

label_fields = {
    "is_declining",
    "trend_direction",
    "trend_pct",
    "decline",
    "label",
    "target",
}

id_fields = {
    "client_hash_id",
    "content_hash_id",
}

future_or_outcome_fields = {
    "imp_last15",
    "imp_prev15",
    "future_impressions",
    "future_clicks",
    "future_position",
}

feature_set = set(feature_cols)

print("=== LEAKAGE CHECK ===")

print("\nLabel-derived fields included:")
print(sorted(feature_set & label_fields))

print("\nIdentifier fields included:")
print(sorted(feature_set & id_fields))

print("\nFuture/outcome fields included:")
print(sorted(feature_set & future_or_outcome_fields))

assert not (feature_set & label_fields), "Label-derived field found in features."
assert not (feature_set & id_fields), "Identifier found in features."
assert not (feature_set & future_or_outcome_fields), "Future/outcome field found in features."

print("\nPASS: no known label-derived, identifier, or future/outcome fields are in the feature vector.")

=== LEAKAGE CHECK ===

Label-derived fields included:
[]

Identifier fields included:
[]

Future/outcome fields included:
[]

PASS: no known label-derived, identifier, or future/outcome fields are in the feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

| Field                                                                               | Reason for exclusion                                                                                                               |
| ----------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| `is_declining`                                                                      | This is the target label, so using it as a feature would directly leak the outcome.                                                |
| `trend_direction`                                                                   | Label-derived information that would not be available before the prediction.                                                       |
| `trend_pct`                                                                         | Directly encodes the change used to define the decline outcome.                                                                    |
| `imp_last15`                                                                        | Represents the later portion of the observation window and therefore contains future information relative to the prediction point. |
| `imp_prev15`                                                                        | Used to construct the decline label and therefore should not be supplied as an independent model feature in this setup.            |
| `client_hash_id`                                                                    | Pseudonymous identifier; useful for grouping/joining but not a legitimate predictive feature.                                      |
| `content_hash_id`                                                                   | Pseudonymous identifier; useful for grouping/joining but not a legitimate predictive feature.                                      |
| Product decision fields such as `health_score`, `priority_score`, and `action_type` | These encode downstream decisions rather than information available independently at prediction time.                              |

The final feature vector therefore contains only the three intended search-performance measurements: `impressions_month`, `clicks_month`, and `avg_position_month`.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature audit

excluded_fields = [
    "is_declining",
    "trend_direction",
    "trend_pct",
    "imp_last15",
    "imp_prev15",
    "client_hash_id",
    "content_hash_id",
    "health_score",
    "priority_score",
    "action_type",
]

print("Final model features:")
for col in feature_cols:
    print(f"  ✓ {col}")

print("\nExcluded fields:")
for col in excluded_fields:
    print(f"  - {col}")

print("\nFeature count:", len(feature_cols))

Final model features:
  ✓ impressions_month
  ✓ clicks_month
  ✓ avg_position_month

Excluded fields:
  - is_declining
  - trend_direction
  - trend_pct
  - imp_last15
  - imp_prev15
  - client_hash_id
  - content_hash_id
  - health_score
  - priority_score
  - action_type

Feature count: 3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.